In [ ]:
import random,os
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

def generate_dim_user(num_users=10000, seed=42):
    random.seed(seed)
    rng = np.random.default_rng(seed)
    pd.set_option("mode.chained_assignment", None)

    log_start = datetime(2026, 7, 1)
    log_end = datetime(2026, 8, 29)

    # 用户注册时段分布规则
    # 1) 老用户：2025-01-01 ~ 2026-06-30，约 60%~70%
    # 2) 新用户：2026-07-01 ~ 2026-08-29，约 30%~40%
    # 3) 禁止：晚于 2026-08-29，0%，避免“还没注册却有行为”的脏数据
    old_ratio = 0.65
    new_ratio = 0.35

    provinces = ["Guangdong", "Beijing", "Shanghai", "Zhejiang", "Jiangsu",
                 "Sichuan", "Hunan", "Hubei", "Shandong", "Shaanxi"]
    cities = {
        "Guangdong": ["Guangzhou", "Shenzhen", "Dongguan", "Foshan"],
        "Beijing": ["Beijing"],
        "Shanghai": ["Shanghai"],
        "Zhejiang": ["Hangzhou", "Ningbo", "Wenzhou"],
        "Jiangsu": ["Nanjing", "Suzhou", "Wuxi"],
        "Sichuan": ["Chengdu", "Chongqing"],
        "Hunan": ["Changsha", "Zhuzhou"],
        "Hubei": ["Wuhan", "Yichang"],
        "Shandong": ["Jinan", "Qingdao"],
        "Shaanxi": ["Xi'an", "Baoji"]
    }

    user_levels = ["free", "standard", "vip"]
    user_types = ["low_frequency", "normal", "high_frequency"]
    # genders = ["male", "female"]
    age_groups = ["18-24", "25-34", "35-44", "45-54", "55+"]
    channels = ["app_store", "wechat_mini_app", "google_play", "sina_weibo", "rednote"]

    male_prob = rng.uniform(0.35, 0.65)                      # 放这里：循环外，只算一次
    gender_arr = np.where(rng.random(num_users) < male_prob, "male", "female")  # 一次性生成所有用户的性别

    records = []

    for i in range(1, num_users + 1):
        user_id = f"user_{i:06d}"

        user_group = random.choices(
            ["old_user", "new_user"],
            weights=[old_ratio, new_ratio],
            k=1
        )[0]

        if user_group == "old_user":
            # 老用户：早于日志开始日期，且主要集中在 2025 ~ 2026-06-30
            old_start = datetime(2025, 1, 1)
            old_end = log_start - timedelta(days=1)
            delta_days = (old_end - old_start).days
            register_time = old_start + timedelta(
                days=random.randint(0, delta_days),
                hours=random.randint(0, 23),
                minutes=random.randint(0, 59)
            )
        else:
            # 新用户：落在 2026-07-01 ~ 2026-08-15
            new_end = datetime(2026, 8, 15)
            delta_days = (new_end - log_start).days
            register_time = log_start + timedelta(
                days=random.randint(0, delta_days),
                hours=random.randint(0, 23),
                minutes=random.randint(0, 59)
            )

        # 绝对禁止晚于日志结束日期的注册时间
        if register_time > log_end:
            register_time = log_end

        province = random.choice(provinces)
        city = random.choice(cities.get(province, [province]))

        user_level = random.choices(
            user_levels,
            weights=[0.55, 0.30, 0.15],
            k=1
        )[0]

        user_type = random.choices(
            user_types,
            weights=[0.25, 0.50, 0.25],
            k=1
        )[0]

        gender = gender_arr[i - 1]
        age_group = random.choice(age_groups)
        is_active = random.choices([True, False], weights=[0.68, 0.32], k=1)[0]
        register_channel = random.choice(channels)

        records.append({
            "user_id": user_id,
            "register_time": register_time.strftime("%Y-%m-%d %H:%M:%S"),
            "user_group": user_group,          # 老用户 / 新用户
            "user_level": user_level,          # free / standard / vip
            "user_type": user_type,            # low_frequency / normal / high_frequency
            "gender": gender,
            "age_group": age_group,
            "province": province,
            "city": city,
            "is_active": is_active,
            "register_channel": register_channel,
        })

    df = pd.DataFrame(records)

    df = df[
        ["user_id", "register_time", "user_group", "user_level", "user_type",
         "gender", "age_group", "province", "city",
         "is_active", "register_channel"]
    ]

    os.makedirs("output_data", exist_ok=True)
    df.to_parquet("output_data/01_dim_user.parquet", index=False)

    print(f"✅ 01_dim_user parquet saved: {len(df)} rows")
    print(df.head(5).to_string(index=False))
    return df


# 主程序
generate_dim_user(num_users=10000)

✅ 01_dim_user parquet saved: 10000 rows
    user_id       register_time user_group user_level      user_type gender age_group  province     city  is_active register_channel
user_000001 2025-01-26 23:17:00   old_user       free  low_frequency   male       55+  Zhejiang Hangzhou       True       sina_weibo
user_000002 2025-04-06 06:14:00   old_user   standard         normal female       55+  Shandong    Jinan       True       sina_weibo
user_000003 2025-01-07 05:44:00   old_user       free  low_frequency female     35-44     Hunan  Zhuzhou       True       sina_weibo
user_000004 2025-12-19 19:16:00   old_user       free high_frequency   male     45-54 Guangdong   Foshan       True      google_play
user_000005 2026-08-09 11:36:00   new_user       free         normal female     35-44  Zhejiang  Wenzhou      False  wechat_mini_app


,user_id,register_time,user_group,user_level,user_type,gender,age_group,province,city,is_active,register_channel
0,user_000001,2025-01-26 23:17:00,old_user,free,low_frequency,male,55+,Zhejiang,Hangzhou,True,sina_weibo
1,user_000002,2025-04-06 06:14:00,old_user,standard,normal,female,55+,Shandong,Jinan,True,sina_weibo
2,user_000003,2025-01-07 05:44:00,old_user,free,low_frequency,female,35-44,Hunan,Zhuzhou,True,sina_weibo
3,user_000004,2025-12-19 19:16:00,old_user,free,high_frequency,male,45-54,Guangdong,Foshan,True,google_play
4,user_000005,2026-08-09 11:36:00,new_user,free,normal,female,35-44,Zhejiang,Wenzhou,False,wechat_mini_app
...,...,...,...,...,...,...,...,...,...,...,...
9995,user_009996,2026-07-11 11:48:00,new_user,standard,low_frequency,male,18-24,Beijing,Beijing,True,rednote
9996,user_009997,2026-07-01 11:13:00,new_user,standard,normal,male,55+,Hubei,Wuhan,True,sina_weibo
9997,user_009998,2026-08-12 03:49:00,new_user,standard,low_frequency,male,55+,Hubei,Wuhan,True,wechat_mini_app
9998,user_009999,2026-06-27 07:48:00,old_user,free,low_frequency,male,18-24,Hubei,Yichang,True,sina_weibo


<!-- pip install pandas==2.2.3 numpy==2.4.6 -->

